# Line of therapy — line-aware contexts

**NOT FOR CLINICAL USE.** Population/trial-level forward simulation only. Executed in CI (nbmake).

The spec's context library is indexed *by tumor type and line of therapy*. NSCLC now has both a first-line and a second-line context (baseline, OS + PFS survival links, eligible TGI models). Survival matching is **line-aware**: a second-line simulation never silently borrows a first-line survival model — a real correctness fix — and second-line prognosis is shorter, as expected.

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import onkos

ds = onkos.load()
t = np.linspace(0, 208, 417)
for line in ['first', 'second']:
    cmp = onkos.compare(ds, purpose='tgi', context={'tumor_type': 'NSCLC', 'line': line}, drug_effect=1.0, t=t)
    ids = [tr.record_id for tr in cmp.included]
    print(f'NSCLC {line}: {len(cmp.included)} models {ids} | median OS range {tuple(round(m,1) for m in cmp.median_os_range)}')
    assert len(cmp.included) >= 2

In [ ]:
# Same model, two lines: second-line survival is shorter (line-aware links).
first = onkos.simulate(ds, 'tgi_metrics.wang_2009.biexponential', context={'tumor_type': 'NSCLC', 'line': 'first'}, drug_effect=1.0, t=t)
second = onkos.simulate(ds, 'tgi_metrics.wang_2009.biexponential', context={'tumor_type': 'NSCLC', 'line': 'second'}, drug_effect=1.0, t=t)
print('1L median OS/PFS:', round(first.median_os, 1), '/', round(first.median_pfs, 1))
print('2L median OS/PFS:', round(second.median_os, 1), '/', round(second.median_pfs, 1))
assert second.median_os < first.median_os and second.median_pfs < first.median_pfs
plt.plot(t, first.os_curve, label='1L OS'); plt.plot(t, second.os_curve, label='2L OS')
plt.axhline(0.5, ls=':', color='grey'); plt.xlabel('weeks'); plt.ylabel('survival'); plt.legend(); plt.title('NSCLC OS by line');

In [ ]:
# A line with no curated survival link gets no survival curve (honest), not a borrowed one.
third = onkos.simulate(ds, 'tgi_metrics.wang_2009.biexponential', context={'tumor_type': 'NSCLC', 'line': 'third'}, drug_effect=1.0, t=t)
print('third-line survival endpoints:', list(third.survival))
assert third.survival == {}